In [6]:
pip install undetected-chromedriver

  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Created wheel for undetected-chromedriver: filename=undetected_chromedriver-3.5.5-py3-none-any.whl size=47214 sha256=5078f2d426db99e18af39ac9ebaa9310b9e9c36e1c1b67b732b39af799c6a10a
  Stored in directory: c:\users\mr\appdata\local\pip\cache\wheels\7a\5f\c1\06f68421cc7172ef51504631252870bcb3a2fdf3b6a025f362
Successfully built undetected-chromedriver

   ---------------------------------------- 0/2 [websockets]
   ---------------------------------------- 0/2 [websockets]
   -------------------- ------------------- 1/2 [undetected-chromedriver]
   ---------------------------------------- 2/2 [undetected-chromedriver]

Note: you may need to restart the kernel to u

In [1]:
#!/usr/bin/env python3
"""Scrape the NSE option-chain page for an index and return a tidy DataFrame.

The flow is:

1. ``NSEBrowserEngine`` spins up a headless Chrome session (and, importantly,
   warms up cookies against the NSE home page first, since the option-chain
   endpoint often returns nothing on a cold, direct hit).
2. ``parse_nse_html`` turns the rendered HTML into an :class:`OptionChainData`
   object holding the underlying price and a per-strike OI / LTP table.
3. ``get_nearest_strikes`` trims that table to a band of strikes around the
   at-the-money (ATM) strike.

All prices are rounded to a fixed number of decimals and all open-interest
columns are stored as integers, so the resulting DataFrame is clean.
"""

from __future__ import annotations

import logging
import re
import time
from datetime import datetime, time as dtime, timedelta, timezone
from dataclasses import dataclass
from typing import Optional

import pandas as pd
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.common.exceptions import TimeoutException, WebDriverException
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import WebDriverWait
from webdriver_manager.chrome import ChromeDriverManager

# --------------------------------------------------------------------------- #
# Logging
# --------------------------------------------------------------------------- #
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
)
logger = logging.getLogger(__name__)

# --------------------------------------------------------------------------- #
# Configuration constants
# --------------------------------------------------------------------------- #
NSE_BASE_URL: str = "https://www.nseindia.com"
OPTION_CHAIN_URL: str = "https://www.nseindia.com/option-chain"
TARGET_SYMBOL: str = "NIFTY"

DEFAULT_RENDER_TIMEOUT: int = 15   # seconds to wait for the table to render
WARMUP_PAUSE: int = 2              # seconds to let cookies settle on the home page
STRIKES_EACH_SIDE: int = 10        # strikes kept above and below ATM

PRICE_DECIMALS: int = 2            # rounding precision for displayed prices

# NSE trades Monday-Friday, 09:15-15:30 India Standard Time. IST has no DST,
# so a fixed UTC+5:30 offset is exact and needs no tzdata dependency.
IST = timezone(timedelta(hours=5, minutes=30))
MARKET_OPEN = dtime(9, 15)
MARKET_CLOSE = dtime(15, 30)

USER_AGENT: str = (
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
    "(KHTML, like Gecko) Chrome/126.0.0.0 Safari/537.36"
)

# DOM identifiers on the NSE page.
PRICE_SPAN_ID: str = "equity_underlyingVal"
OPTION_TABLE_ID: str = "optionChainTable-indices"

# Zero-based <td> indices within a data row. The table interleaves Call columns
# (left), the Strike (centre) and Put columns (right). These were reverse
# engineered from the live page and may need updating if NSE changes its markup.
COL_CALL_OI: int = 1
COL_CALL_CHNG_OI: int = 2
COL_CALL_LTP: int = 5
COL_STRIKE: int = 11
COL_PUT_LTP: int = 17
COL_PUT_CHNG_OI: int = 20
COL_PUT_OI: int = 21

# A valid data row must contain at least up to the last index we read, so it
# needs (COL_PUT_OI + 1) cells. (The original `>= 21` guard was off by one and
# could raise IndexError when a row had exactly 21 cells.)
MIN_REQUIRED_COLS: int = COL_PUT_OI + 1

# Columns that are whole-number counts vs. columns that are prices.
_INT_COLUMNS = ("Call OI", "Call Chng OI", "Strike Price", "Put Chng OI", "Put OI")
_PRICE_COLUMNS = ("Call LTP", "Put LTP")

# Matches a number with optional thousands commas and optional decimals.
_NUMBER_RE = re.compile(r"\d[\d,]*(?:\.\d+)?")


@dataclass
class OptionChainData:
    """Standardised container for parsed option-chain data."""

    symbol: str
    underlying_price: float
    dataframe: pd.DataFrame


class NSEBrowserEngine:
    """Manage the Selenium WebDriver lifecycle via a context manager."""

    def __init__(self, headless: bool = True) -> None:
        self.headless = headless
        self.driver: Optional[webdriver.Chrome] = None

    def __enter__(self) -> "NSEBrowserEngine":
        options = Options()
        if self.headless:
            # "new" headless renders much more like real Chrome (Chrome 109+).
            options.add_argument("--headless=new")
        # Flags that keep headless Chrome stable on servers / CI.
        options.add_argument("--disable-gpu")
        options.add_argument("--no-sandbox")
        options.add_argument("--disable-dev-shm-usage")
        options.add_argument("--window-size=1920,1080")
        options.add_argument(f"user-agent={USER_AGENT}")
        # Reduce the most obvious "I am automated" tells.
        options.add_argument("--disable-blink-features=AutomationControlled")
        options.add_experimental_option("excludeSwitches", ["enable-automation"])
        options.add_experimental_option("useAutomationExtension", False)

        service = Service(ChromeDriverManager().install())
        self.driver = webdriver.Chrome(service=service, options=options)
        self.driver.set_page_load_timeout(60)
        return self

    def __exit__(self, exc_type, exc_val, exc_tb) -> None:
        if self.driver:
            self.driver.quit()
            logger.info("Browser engine shutdown successfully.")

    def warm_up(self, base_url: str = NSE_BASE_URL, pause: int = WARMUP_PAUSE) -> None:
        """Visit the NSE home page so the server hands us its session cookies.

        NSE frequently returns an empty / blocked option-chain page on a cold,
        direct request; loading the home page first usually fixes that.
        """
        logger.info("Warming up session at %s ...", base_url)
        self.driver.get(base_url)
        time.sleep(pause)

    def fetch_page_source(
        self, url: str, timeout: int = DEFAULT_RENDER_TIMEOUT
    ) -> str:
        """Navigate to ``url`` and return the page source once the table renders.

        Uses an explicit wait for the option-chain table instead of a blind
        ``time.sleep``, so we proceed as soon as the data is present (and never
        wait longer than ``timeout``).
        """
        logger.info("Navigating to %s ...", url)
        self.driver.get(url)
        try:
            WebDriverWait(self.driver, timeout).until(
                EC.presence_of_element_located((By.ID, OPTION_TABLE_ID))
            )
            logger.info("Option chain table detected.")
        except TimeoutException:
            logger.warning(
                "Timed out after %ss waiting for the option chain table; "
                "returning whatever loaded.",
                timeout,
            )
        return self.driver.page_source


def clean_number(text: str) -> float:
    """Convert a possibly comma-formatted numeric string to a float.

    Returns ``0.0`` for blanks, dashes, or anything unparseable.
    """
    text = text.strip()
    if text in ("-", "", "None"):
        return 0.0
    try:
        return float(text.replace(",", ""))
    except ValueError:
        return 0.0


def _tidy_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    """Round price columns and cast open-interest columns to integers."""
    for column in _INT_COLUMNS:
        if column in df.columns:
            # clean_number never returns NaN, so rounding to int is safe.
            df[column] = df[column].round(0).astype(int)
    for column in _PRICE_COLUMNS:
        if column in df.columns:
            df[column] = df[column].round(PRICE_DECIMALS)
    return df


def _extract_price(text: str) -> float:
    """Pull the underlying price out of the price-span text.

    The span text can take several shapes ("22,612.35", "NIFTY 22,612.35",
    or "NIFTY 22,612.35 As on 15-Jun-2026"), so instead of trusting token
    position we take the largest positive number in the text. An index spot
    price is far larger than any stray figure (lot size, a year, etc.), which
    makes this reliable in practice.
    """
    candidates = [clean_number(token) for token in _NUMBER_RE.findall(text)]
    positive = [value for value in candidates if value > 0]
    return max(positive) if positive else 0.0


def parse_nse_html(html: str, symbol: str = TARGET_SYMBOL) -> Optional[OptionChainData]:
    """Extract the underlying price and the per-strike OI / LTP table."""
    soup = BeautifulSoup(html, "html.parser")

    # --- Underlying (spot) price ---
    price_span = soup.find("span", id=PRICE_SPAN_ID)
    if price_span is None:
        logger.error("Could not locate underlying price span. NSE might be blocking.")
        return None

    # The span text looks like "NIFTY 22,600.50 ..."; pull the price robustly.
    live_price = round(_extract_price(price_span.get_text(" ", strip=True)), PRICE_DECIMALS)
    if live_price <= 0:
        logger.error("Parsed a non-positive underlying price (%s); aborting.", live_price)
        return None

    # --- Option chain table ---
    table = soup.find("table", id=OPTION_TABLE_ID)
    if table is None:
        logger.error("Could not find the option chain table.")
        return None

    body = table.find("tbody")
    if body is None:
        logger.error("Option chain table has no <tbody>.")
        return None

    scraped_data: list[dict[str, float]] = []
    for row in body.find_all("tr"):
        cols = row.find_all("td")
        if len(cols) < MIN_REQUIRED_COLS:
            continue  # header, spacer, or partial row

        strike_text = cols[COL_STRIKE].text.strip()
        if not strike_text or strike_text == "-":
            continue

        scraped_data.append(
            {
                "Call OI": clean_number(cols[COL_CALL_OI].text),
                "Call Chng OI": clean_number(cols[COL_CALL_CHNG_OI].text),
                "Call LTP": clean_number(cols[COL_CALL_LTP].text),
                "Strike Price": clean_number(strike_text),
                "Put LTP": clean_number(cols[COL_PUT_LTP].text),
                "Put Chng OI": clean_number(cols[COL_PUT_CHNG_OI].text),
                "Put OI": clean_number(cols[COL_PUT_OI].text),
            }
        )

    if not scraped_data:
        logger.warning("Table was found but no data rows were extracted.")
        return None

    df = _tidy_dataframe(pd.DataFrame(scraped_data))
    return OptionChainData(symbol=symbol, underlying_price=live_price, dataframe=df)


def get_nearest_strikes(
    chain_data: OptionChainData, step: int = STRIKES_EACH_SIDE
) -> pd.DataFrame:
    """Return ``step`` strikes above and below the ATM strike (ATM included)."""
    df = chain_data.dataframe.copy()
    live_price = chain_data.underlying_price

    # The ATM strike is the one closest to the live underlying price.
    atm_index = int((df["Strike Price"] - live_price).abs().idxmin())
    atm_strike = df.loc[atm_index, "Strike Price"]
    logger.info("Underlying: %s | Closest ATM strike: %s", live_price, atm_strike)

    start_index = max(0, atm_index - step)
    end_index = min(len(df), atm_index + step + 1)
    return df.iloc[start_index:end_index].reset_index(drop=True)


def is_market_open(now: Optional[datetime] = None) -> bool:
    """Return ``True`` only during NSE trading hours: Mon-Fri, 09:15-15:30 IST.

    Pass a timezone-aware ``now`` to test specific moments; it defaults to the
    current time in IST. NSE *trading holidays* are not handled here — add a
    holiday-date check inside this function if you need that.
    """
    now = now or datetime.now(IST)
    if now.weekday() >= 5:          # 5 = Saturday, 6 = Sunday
        return False
    return MARKET_OPEN <= now.time() <= MARKET_CLOSE


# --------------------------------------------------------------------------- #
# Main execution
# --------------------------------------------------------------------------- #
def main() -> None:
    """Fetch, parse, and display the option chain for ``TARGET_SYMBOL``."""
    if not is_market_open():
        logger.info(
            "Market is closed right now (%s IST). NSE trades Mon-Fri "
            "09:15-15:30 IST — exiting without scraping.",
            datetime.now(IST).strftime("%Y-%m-%d %H:%M:%S"),
        )
        return

    try:
        with NSEBrowserEngine(headless=True) as browser:
            browser.warm_up()
            raw_html = browser.fetch_page_source(OPTION_CHAIN_URL)
    except WebDriverException as exc:
        logger.error("Browser/WebDriver failure: %s", exc)
        return

    option_data = parse_nse_html(raw_html, symbol=TARGET_SYMBOL)
    if option_data is None:
        logger.error("No option-chain data could be parsed.")
        return

    final_df = get_nearest_strikes(option_data, step=STRIKES_EACH_SIDE)

    # Make sure wide frames print fully.
    pd.set_option("display.max_columns", None)
    pd.set_option("display.width", 1000)

    print(f"\nCurrent {TARGET_SYMBOL} Price: {option_data.underlying_price}")
    print("\nFiltered Option Chain Data (with OI):")
    print(final_df.to_string(index=False))

    # With the data in hand, simple analytics are one-liners.
    print(f"\nHighest Call OI in range: {final_df['Call OI'].max():,}")
    print(f"Highest Put OI in range:  {final_df['Put OI'].max():,}")


if __name__ == "__main__":
    main()

2026-06-16 19:52:53,690 - INFO - Market is closed right now (2026-06-16 19:52:53 IST). NSE trades Mon-Fri 09:15-15:30 IST — exiting without scraping.


In [17]:
final_df

,Call OI,Call Chng OI,Call LTP,Strike Price,Put LTP,Put Chng OI,Put OI
0,24900.0,-6753.0,488.00,23500.0,0.55,65838.0,225294.0
1,8558.0,-801.0,440.10,23550.0,0.60,48055.0,96301.0
2,14143.0,-4639.0,390.55,23600.0,0.60,35113.0,146809.0
3,5229.0,-2302.0,340.20,23650.0,0.65,83960.0,131168.0
4,16384.0,-7676.0,289.95,23700.0,0.70,95308.0,182733.0
5,10054.0,-3972.0,240.35,23750.0,0.75,142774.0,193258.0
6,44700.0,-14230.0,189.65,23800.0,0.85,152217.0,255527.0
7,46080.0,-5804.0,140.25,23850.0,0.95,192516.0,246798.0
8,200903.0,56224.0,91.20,23900.0,2.10,664583.0,740051.0
9,363112.0,249118.0,46.30,23950.0,7.20,879157.0,912810.0


In [18]:
pip install schedule

Note: you may need to restart the kernel to use updated packages.
